<a href="https://colab.research.google.com/github/SohamManik/llm-huggingface/blob/main/LLM14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade transformers huggingface_hub

In [2]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from transformers import AutoTokenizer

# Install the latest version of transformers and huggingface_hub to ensure compatibility


mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1", trust_remote_code = True )
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat" , trust_remote_code=True)
smol_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct", trust_remote_code = True)



messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Who won the world series in 2020?"},

]

In [4]:
mistral_chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
        "{% set loop_messages = messages[1:] %}"
        "{% set system_message = messages[0]['content'] %}"
    "{% else %}"
        "{% set loop_messages = messages %}"
        "{% set system_message = false %}"
    "{% endif %}"
    "{% for message in loop_messages %}"
        "{% if (message['role'] == 'user') and (loop.index == 1) %}"
            "{{ '<s>[INST] ' }}"
            "{% if system_message != false %}"
                "{{ '<<SYS>>\\n' + system_message + '\\n<</SYS>>\\n\\n' }}"
            "{% endif %}"
            "{{ message['content'] + ' [/INST]' }}"
        "{% elif message['role'] == 'user' %}"
            "{{ '[INST] ' + message['content'] + ' [/INST]' }}"
        "{% elif message['role'] == 'assistant' %}"
            "{{ ' '  + message['content'] + '</s>' }}"
        "{% endif %}"
    "{% endfor %}"
)
mistral_chat = mistral_tokenizer.apply_chat_template(messages, tokenize=False, chat_template=mistral_chat_template)

In [5]:
mistral_chat

'<s>[INST] <<SYS>>\nYou are a helpful assistant\n<</SYS>>\n\nWho won the world series in 2020? [/INST]'

In [6]:
%pip install trl
%pip install datasets
%pip install torch

In [7]:
from datasets import load_dataset

from trl import SFTConfig , SFTTrainer
import torch

In [9]:
from transformers import AutoModelForCausalLM

In [21]:
from datasets import load_dataset

from trl import SFTConfig , SFTTrainer
import torch

dataset = load_dataset("HuggingFaceTB/smoltalk" , "all")

model_name = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path = model_name)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path = model_name)

# Set the chat template for the tokenizer
tokenizer.chat_template = mistral_chat_template

# Removed: model , tokenizer = setup_chat_format(model = model , tokenizer = tokenizer) as it does not exist in trl 1.5.1

training_args = SFTConfig(
    output_dir = "./sft_output",
    max_steps = 1000,
    per_device_train_batch_size = 4,
    learning_rate = 5e-5,
    logging_steps = 50 ,
    save_steps = 100,
    eval_strategy = "steps",
    eval_steps = 50,




)


trainer = SFTTrainer(
    model = model ,
    args = training_args ,
    train_dataset = dataset["train"] ,
    eval_dataset = dataset["test"], # Added back eval_dataset
    processing_class = tokenizer, # Reverted to 'processing_class' as 'tokenizer' is not accepted
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Tokenizing eval dataset:   0%|          | 0/54948 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9441 > 8192). Running this sequence through the model will result in indexing errors


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss
